In [27]:
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA
import numpy as np
from sentence_transformers import SentenceTransformer

import os
import datetime
import sqlite3

import optuna
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import cross_val_score
from sklearn.metrics import make_scorer, root_mean_squared_error



In [2]:
raw_train = pd.read_csv("../data/01_raw/train.csv", parse_dates= ['DateTimeOfAccident', 'DateReported' ])
raw_test = pd.read_csv("../data/01_raw/test.csv", parse_dates= ['DateTimeOfAccident', 'DateReported' ])


In [3]:
df_train = raw_train.copy()


# Feature engineering

## Parameters config

In [170]:
target_name = 'UltimateIncurredClaimCost'
multiplier = 6
impute_columns = 'MaritalStatus'
impute_fill_value = 'U'
emb_model_name = "Qwen/Qwen3-Embedding-0.6B"
batch_size = 32
load_embeddings_path = "../data/02_intermediate/embeddings/desc_embeddings.db"
save_embeddings_path = "../data/02_intermediate/embeddings/"
load_pca_path = None
save_pca_path = "../data/02_intermediate/pca/"
n_components = 100
to_drop = ["ClaimDescription","ClaimNumber","DateTimeOfAccident","DateReported"]
cat_features = ['Gender', 'MaritalStatus', 'PartTimeFullTime']
encoded_cat_features = ['Gender_F',
 'Gender_M',
 'Gender_U',
 'MaritalStatus_M',
 'MaritalStatus_S',
 'MaritalStatus_U',
 'PartTimeFullTime_F',
 'PartTimeFullTime_P']


In [171]:
def DropLargeOutliers(multiplier=multiplier):

    class _DropLargeOutliers(BaseEstimator, TransformerMixin):
        def __init__(self, multiplier):
            self.multiplier = multiplier

        def fit(self, X, y):
            y = pd.Series(y)
            q1 = y.quantile(0.25)
            q3 = y.quantile(0.75)
            self.upper_ = q3 + self.multiplier * (q3 - q1)
            self.mask_ = y <= self.upper_
            return self

        def transform(self, X):
            X = pd.DataFrame(X).iloc[self.mask_.values].copy()
            return X

    return _DropLargeOutliers(multiplier)


In [177]:
import time
from datetime import datetime
from tracemalloc import start

def ColumnDropper(to_drop=None):
    
    class _ColumnDropper(BaseEstimator, TransformerMixin):
        def __init__(self, to_drop):
            self.to_drop=to_drop
        
        def fit(self, X, y=None):
            self.fitted_ = True
            return self 
        
        def transform(self, X):
            return X.drop(self.to_drop,axis=1).reset_index(drop=True)
    
    return _ColumnDropper(to_drop)


# date features

def ReportDelayHours():
    
    class _ReportDelayHours(BaseEstimator, TransformerMixin):
        
        def __init__(self):
            return
        
        def fit(self, X, y=None):
            self.fitted_ = True
            return self
        
        def transform(self, X):
            X = X.copy()
            X['report_delay']= X['DateReported'] - X['DateTimeOfAccident']
            X['report_delay_hours'] = X['report_delay'].dt.total_seconds() / 3600

            # reset index to concatenate correctly with FeatureUnion to pca embeddings (our row key is ClaimNumber)
            X.reset_index(inplace=True)
            
            return X[['report_delay_hours']]
    
    return _ReportDelayHours()

def AccidentTime():

    class _AccidentTime(BaseEstimator, TransformerMixin):

        def __init__(self):
            return
        
        def fit(self, X, y=None):
            self.fitted_ = True
            return self
        
        def transform(self, X):
            X = X.copy()
            X['accident_year'] = X['DateTimeOfAccident'].dt.year
            X['accident_month'] = X['DateTimeOfAccident'].dt.month
            X['accident_day'] = X['DateTimeOfAccident'].dt.day
            X['accident_hour'] = X['DateTimeOfAccident'].dt.hour

            # reset index to concatenate correctly with FeatureUnion to pca embeddings (our row key is ClaimNumber)
            X.reset_index(inplace=True)

            return X[['accident_year','accident_month','accident_day','accident_hour']]
    
    return _AccidentTime()


def ReportTime():

    class _ReportTime(BaseEstimator, TransformerMixin):

        def __init__(self):
            return
        
        def fit(self, X, y=None):
            self.fitted_ = True
            return self
        
        def transform(self, X):
            X = X.copy()
            X['reported_year'] = X['DateReported'].dt.year
            X['reported_month'] = X['DateReported'].dt.month
            X['reported_day'] = X['DateReported'].dt.day

            # reset index to concatenate correctly with FeatureUnion to pca embeddings (our row key is ClaimNumber)
            X.reset_index(inplace=True)

            return X[['reported_year','reported_month','reported_day']]
    
    return _ReportTime()

# Claim description word count

def ClaimDescriptionWordCount():

    class _ClaimDescriptionWordCount(BaseEstimator, TransformerMixin):

        def __init__(self):
            return
        
        def fit(self, X, y=None):
            self.fitted_ = True
            return self
        
        def transform(self, X):
            X = X.copy()
            X["ClaimDescription_word_count"] = X['ClaimDescription'].apply(lambda x:len(str(x).split()))
            
            # reset index to concatenate correctly with FeatureUnion to pca embeddings (our row key is ClaimNumber)
            X.reset_index(inplace=True)

            return X[["ClaimDescription_word_count"]]
    
    return _ClaimDescriptionWordCount()


def ColumnSelector(column):

    class _ColumnSelector(BaseEstimator, TransformerMixin):

        def __init__(self, column):
            self.column = column

        def fit(self, X, y=None):
            self.fitted_ = True
            return self

        def transform(self, X):
            return X[self.column]
    
    return _ColumnSelector(column)


def SentenceEmbedding(model_name=emb_model_name, 
                      batch_size=batch_size,
                      load_embeddings_path=load_embeddings_path,
                      save_embeddings_path=save_embeddings_path, 
                      verbose=True):

    class _SentenceEmbedding(BaseEstimator, TransformerMixin):

        def __init__(self, model_name, batch_size, load_embeddings_path, save_embeddings_path, verbose):
            self.model_name = model_name
            self.batch_size = batch_size
            self.load_embeddings_path = load_embeddings_path
            self.save_embeddings_path = save_embeddings_path
            self.verbose = verbose

        def _model_tag(self):
            return self.model_name.replace("/", "_").replace("\\", "_")
        
        def _log(self, msg):
            if self.verbose:
                print(f"[SentenceEmbedding] {msg}")

        def _version_path(self):
            ts = datetime.now().strftime("%Y%m%d_%H%M%S")
            model_tag = self._model_tag()
            path = os.path.join(self.save_embeddings_path, f"desc_embeddings_{model_tag}_{ts}.db")
            return path
        
        
        def fit(self, X, y=None):
            # Skip fitting if loading precomputed embeddings
            if self.load_embeddings_path is not None and os.path.exists(self.load_embeddings_path):
                self._log(f"Skipping fit (loading embeddings from {self.load_embeddings_path})")
                return self
            
            self._log(f"Loading model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            self.fitted_ = True
            return self
        
        def transform(self, X):
            
            # Option 1: load existing embeddings
            if self.load_embeddings_path is not None and os.path.exists(self.load_embeddings_path):
                self._log(f"Loading embeddings from {self.load_embeddings_path} ...")
                conn = sqlite3.connect(self.load_embeddings_path)
                emb_df = pd.read_sql(
                    "SELECT * FROM desc_embeddings",
                    conn
                    )
                conn.close()

                self._log(f"Loaded embeddings shape: {emb_df.shape}")

                # filter by ClaimNumber in dataset
                emb_df_filtered = (X[["ClaimNumber"]].merge(emb_df, on="ClaimNumber", how="left"))
                emb = emb_df_filtered.drop(["ClaimNumber"], axis=1)

                # replace NaNs and infs
                embeddings = np.nan_to_num(emb, nan=0.0, posinf=0.0, neginf=0.0)

                return embeddings
            
            # Option 2: compute embeddings
            self._log("Computing embeddings...")

            # input format to sentence transformer is list of strings
            X_list = pd.Series(X).fillna("").astype(str).tolist()
            
            start = time.time()
            emb = self.model.encode(
            X_list,
            batch_size=self.batch_size,
            show_progress_bar=True
            )
            self._log(f"Embedding done in {time.time() - start:.2f}s")
            
            embeddings = emb.to_numpy(dtype=np.float32)

            # Save embeddings dataframe as a SQL table to versionned path
            self._log("Saving embeddings...")
            df_embeddings = X[["ClaimNumber"]].join(pd.DataFrame(embeddings))
            version_path = self._version_path()
            
            conn = sqlite3.connect(version_path)

            df_embeddings.to_sql(
                name = "desc_embeddings",
                con=conn,
                if_exists="fail",
                index=False
            )

            conn.close()

            self._log(f"Saved embeddings to {version_path}")

            return embeddings
        


    return _SentenceEmbedding(model_name, batch_size, load_embeddings_path,
                      save_embeddings_path, 
                      verbose)

def PCATransformer(n_components=n_components, 
                   verbose=True):

    class _PCATransformer(BaseEstimator, TransformerMixin):

        def __init__(self, n_components, verbose):
            self.n_components = n_components
            self.verbose = verbose

        def _log(self, msg):
            if self.verbose:
                print(f"[PCA] {msg}")

        def _model_tag(self):
            return f"pca_{self.n_components}"
        

        def fit(self, X, y=None):
            
            self._log(f"Fitting PCA (n_components={self.n_components})...")
            start = time.time()

            self.pca = PCA(n_components=self.n_components)
            self.pca.fit(X)

            self._log(f"PCA fitted in {time.time() - start:.2f}s")
            self.fitted_ = True

            return self

        def transform(self, X):

            # Compute PCA
            self._log("Transforming embeddings with PCA...")

            start = time.time()
            X_pca = self.pca.transform(X)

            self._log(f"PCA done in {time.time() - start:.2f}s")
            self._log(f"PCA output shape: {X_pca.shape}")

            return X_pca

    return _PCATransformer(n_components , verbose)


In [178]:
from sklearn import set_config

set_config(transform_output="pandas") # to get dataframe as Transformer output

date_feature_eng = FeatureUnion(
    [
        ('add_report_delay_days', ReportDelayHours()),
        ('add_accident_time_features', AccidentTime()),
        ('add_report_time_features', ReportTime())
    ]
)
desc_embedding_pca = Pipeline([
    ("select_desc_column", ColumnSelector(column=["ClaimNumber", "ClaimDescription"])),
    ("embed_desc", SentenceEmbedding(
        model_name="Qwen/Qwen3-Embedding-0.6B",
        batch_size=32
    )),
    ("pca_embed_desc", PCA(n_components=n_components))
])

claim_desc_feature_eng = FeatureUnion(
    [
        ('add_desc_word_count', ClaimDescriptionWordCount()),
        ('add_desc_embedding_pca',desc_embedding_pca )
    ]
)

impute_MaritalStatus = ColumnTransformer(
    transformers=[
        (
            'impute_MaritalStatus',
            SimpleImputer(strategy='constant', fill_value=impute_fill_value),
            ['MaritalStatus'] 
        )
    ],
    remainder='passthrough',
    verbose_feature_names_out=False
)

encode_cat_features = ColumnTransformer(
    transformers=[
        (
            'encode_cat_features',
            OneHotEncoder(handle_unknown='ignore', sparse_output=False), 
            cat_features
        )
    ],
    remainder='passthrough',
    verbose_feature_names_out=False
)

cat_feature_eng = Pipeline(
    [
        ('impute_MaritalStatus', impute_MaritalStatus),
        ('encode_cat_features', encode_cat_features)
    ]
)

feature_engineering_pipeline = Pipeline(
    [   
        ('cat_feature_eng', cat_feature_eng),
        ('add_new_features', FeatureUnion(
            [
                ('date_fe', date_feature_eng),
                ('claim_desc_fe', claim_desc_feature_eng),
                ('drop_cols', ColumnDropper(to_drop))
            ]
        ) )
       
    ]
)


## Run feature engineering pipeline

In [179]:
feature_engineering_pipeline


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('cat_feature_eng', ...), ('add_new_features', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('impute_MaritalStatus', ...), ('encode_cat_features', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid sea

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    df_train.drop(columns=[target_name]),
    df_train[target_name],
    test_size=0.2,
    random_state=42
)
feature_engineering_pipeline.fit(X_train, y_train)


[SentenceEmbedding] Skipping fit (loading embeddings from ../data/02_intermediate/embeddings/desc_embeddings.db)
[SentenceEmbedding] Loading embeddings from ../data/02_intermediate/embeddings/desc_embeddings.db ...
[SentenceEmbedding] Loaded embeddings shape: (54000, 1025)


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('cat_feature_eng', ...), ('add_new_features', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('impute_MaritalStatus', ...), ('encode_cat_features', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid sea

In [46]:
X_train_fe = feature_engineering_pipeline.transform(X_train)


[SentenceEmbedding] Loading embeddings from ../data/02_intermediate/embeddings/desc_embeddings.db ...
[SentenceEmbedding] Loaded embeddings shape: (54000, 1025)


In [47]:
X_test_fe = feature_engineering_pipeline.transform(X_test)


[SentenceEmbedding] Loading embeddings from ../data/02_intermediate/embeddings/desc_embeddings.db ...
[SentenceEmbedding] Loaded embeddings shape: (54000, 1025)


In [48]:
# check
print(X_train_fe.shape, X_test_fe.shape)


(43200, 124) (10800, 124)


In [12]:
feature_engineering_pipeline.__sklearn_is_fitted__()


True

In [ ]:
# X_train_fe.to_csv("../data/05_model_input/X_train.csv", index=False)
# y_train.to_csv("../data/05_model_input/y_train.csv", index=False)


# Fit a baseline model

In [ ]:
hgb_model = HistGradientBoostingRegressor(
    loss='squared_error',          # similar to RMSE optimization
    learning_rate=0.05,
    max_iter=100,
    max_depth=6,
    min_samples_leaf=20,
    random_state=42,
    categorical_features=encoded_cat_features,
)


In [ ]:
hgb_model.fit(X_train_fe, y_train)


,"loss loss: {'squared_error', 'absolute_error', 'gamma', 'poisson', 'quantile'}, default='squared_error'The loss function to use in the boosting process. Note that the""squared error"", ""gamma"" and ""poisson"" losses actually implement""half least squares loss"", ""half gamma deviance"" and ""half poissondeviance"" to simplify the computation of the gradient. Furthermore,""gamma"" and ""poisson"" losses internally use a log-link, ""gamma""requires ``y > 0`` and ""poisson"" requires ``y >= 0``.""quantile"" uses the pinball loss... versionchanged:: 0.23 Added option 'poisson'... versionchanged:: 1.1 Added option 'quantile'... versionchanged:: 1.3 Added option 'gamma'.",'squared_error'
,"quantile quantile: float, default=NoneIf loss is ""quantile"", this parameter specifies which quantile to be estimatedand must be between 0 and 1.",None
,"learning_rate learning_rate: float, default=0.1The learning rate, also known as *shrinkage*. This is used as amultiplicative factor for the leaves values. Use ``1`` for noshrinkage.",0.05
,"max_iter max_iter: int, default=100The maximum number of iterations of the boosting process, i.e. themaximum number of trees.",100
,"max_leaf_nodes max_leaf_nodes: int or None, default=31The maximum number of leaves for each tree. Must be strictly greaterthan 1. If None, there is no maximum limit.",31
,"max_depth max_depth: int or None, default=NoneThe maximum depth of each tree. The depth of a tree is the number ofedges to go from the root to the deepest leaf.Depth isn't constrained by default.",6
,"min_samples_leaf min_samples_leaf: int, default=20The minimum number of samples per leaf. For small datasets with lessthan a few hundred samples, it is recommended to lower this valuesince only very shallow trees would be built.",20
,"l2_regularization l2_regularization: float, default=0The L2 regularization parameter penalizing leaves with small hessians.Use ``0`` for no regularization (default).",0.0
,"max_features max_features: float, default=1.0Proportion of randomly chosen features in each and every node split.This is a form of regularization, smaller values make the trees weakerlearners and might prevent overfitting.If interaction constraints from `interaction_cst` are present, only allowedfeatures are taken into account for the subsampling... versionadded:: 1.4",1.0
,"max_bins max_bins: int, default=255The maximum number of bins to use for non-missing values. Beforetraining, each feature of the input array `X` is binned intointeger-valued bins, which allows for a much faster training stage.Features with a small number of unique values may use less than``max_bins`` bins. In addition to the ``max_bins`` bins, one more binis always reserved for missing values. Must be no larger than 255.",255
,"categorical_features categorical_features: array-like of {bool, int, str} of shape (n_features) or shape (n_categorical_features,), default='from_dtype'Indicates the categorical features.- None : no feature will be considered categorical.- boolean array-like : boolean mask indicating categorical features.- integer array-like : integer indices indicating categorical features.- str array-like: names of categorical features (assuming the training data has feature names).- `""from_dtype""`: dataframe columns with dtype ""category"" are considered to be categorical features. The input must be an object exposing a ``__dataframe__`` method such as pandas or polars DataFrames to use this feature.For each categorical feature, there must be at most `max_bins` uniquecategories. Negative values for categorical features encoded as numericdtypes are treated as missing values. All categorical values areconverted to floating point numbers. This means that categorical valuesof 1.0 and 1 are treated as the same category.Read more in the :ref:`User Guide ` and:ref:`sphx_glr_auto_examples_ensemble_plot_gradient_boosting_categorical.py`... versionadded:: 0.24.. versionchanged:: 1.2 Added support for feature names... versionchanged:: 1.

In [49]:
y_pred = hgb_model.predict(X_test_fe)
rmse = root_mean_squared_error(y_test, y_pred)
print(rmse)


25497.852897041583


In [13]:
# lgbm_model = LGBMRegressor(
#     #objective='regression_l1',
#     #metric='rmse',
#     #n_estimators=100,
#     #learning_rate=0.05,
#     #num_leaves=31,
#     #max_depth=-1,
#     #min_child_samples=20,
#     #subsample=0.8,
#     #colsample_bytree=0.8,
#     #random_state=42
# )


In [14]:
#lgbm_model.fit(X_train_fe, y_train,
#               eval_metric="rmse",
#               categorical_feature=encoded_cat_features
#)



# Hyperparameters Tuning

## Prepare data: feature engineering

In [180]:
# Feature eng on all train dataset

X_df_train = df_train.drop(columns=[target_name])
y_df_train = df_train[target_name]


In [181]:
# Drop rows with extreme target values

X_df_train = DropLargeOutliers(multiplier=multiplier).fit(X_df_train, y_df_train).transform(X_df_train)
y_df_train = DropLargeOutliers(multiplier=multiplier).fit(X_df_train, y_df_train).transform(y_df_train)


In [182]:
feature_engineering_pipeline.fit(X_df_train, y_df_train)


[SentenceEmbedding] Skipping fit (loading embeddings from ../data/02_intermediate/embeddings/desc_embeddings.db)
[SentenceEmbedding] Loading embeddings from ../data/02_intermediate/embeddings/desc_embeddings.db ...
[SentenceEmbedding] Loaded embeddings shape: (54000, 1025)


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('cat_feature_eng', ...), ('add_new_features', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('impute_MaritalStatus', ...), ('encode_cat_features', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid sea

In [183]:
X_df_train.shape


(51663, 14)

In [184]:
X_df_train_fe = feature_engineering_pipeline.transform(X_df_train)


[SentenceEmbedding] Loading embeddings from ../data/02_intermediate/embeddings/desc_embeddings.db ...
[SentenceEmbedding] Loaded embeddings shape: (54000, 1025)


In [185]:
print(X_df_train_fe.shape, y_df_train.shape)


(51663, 124) (51663, 1)


In [186]:
y_df_train = y_df_train["UltimateIncurredClaimCost"]


In [187]:
y_df_train


0         4748.203388
1         6326.285819
2         2293.949087
3        17786.487170
4         4014.002925
             ...     
53995      480.493308
53996      755.735319
53997      418.178461
53998     2695.225700
53999      934.273548
Name: UltimateIncurredClaimCost, Length: 51663, dtype: float64

## Tuning with CV

In [189]:
from sklearn.model_selection import KFold


optuna.logging.set_verbosity(optuna.logging.FATAL)

def objective(trial, X_train, y_train):

    params = {

        # model capacity
        "max_leaf_nodes": trial.suggest_int("max_leaf_nodes", 31, 255),
        "max_depth": trial.suggest_int("max_depth", 3, 16),

        # learning dynamics
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.15, log=True),
        "max_iter": trial.suggest_int("max_iter", 300, 2000),

        # regularization
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 10, 120),
        "l2_regularization": trial.suggest_float("l2_regularization", 1e-6, 10.0, log=True),

        # stochasticity
        "max_features": trial.suggest_float("max_features", 0.6, 1.0),

        # binning resolution
        "max_bins": trial.suggest_int("max_bins", 128, 255),

        # Objective
        "loss": "squared_error",
        "random_state": 42
    }

    model = HistGradientBoostingRegressor(**params)

    score = cross_val_score(estimator = model, X=X_train, y=y_train, cv=KFold(n_splits=10,  shuffle=True, random_state=1), scoring='neg_mean_squared_error')
    mean_score = score.mean()
    mean_rmse = np.sqrt(-mean_score)

    return mean_rmse

study = optuna.create_study(direction="minimize")

study.optimize(lambda trial: objective(trial, X_df_train_fe, y_df_train), n_trials=100, show_progress_bar=True)


Best trial: 74. Best value: 4428.32: 100%|██████████| 100/100 [1:56:01<00:00, 69.62s/it]


### Test tuning with log transformed target column

In [91]:
from sklearn.compose import TransformedTargetRegressor
from sklearn.model_selection import KFold

# Log Transformation on y

#optuna.logging.set_verbosity(optuna.logging.FATAL)

def objective(trial, X_train, y_train):

    params = {

        # model capacity
        "max_leaf_nodes": trial.suggest_int("max_leaf_nodes", 31, 255),
        "max_depth": trial.suggest_int("max_depth", 3, 16),

        # learning dynamics
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.15, log=True),
        "max_iter": trial.suggest_int("max_iter", 300, 2000),

        # regularization
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 10, 120),
        "l2_regularization": trial.suggest_float("l2_regularization", 1e-6, 10.0, log=True),

        # stochasticity
        "max_features": trial.suggest_float("max_features", 0.6, 1.0),

        # binning resolution
        "max_bins": trial.suggest_int("max_bins", 128, 255),

        # Objective
        "loss": "squared_error",
        "random_state": 42
    }

    base_model = HistGradientBoostingRegressor(**params)

    # metric = root_mean_squared_error(model.predict(X_val), y_val)
    
    # Wrap model with log1p transform on y
    model = TransformedTargetRegressor(
        regressor=base_model,
        func=np.log1p,
        inverse_func=np.expm1
    )

    score = cross_val_score(estimator = model, X=X_train, y=y_train, cv=KFold(n_splits=10,  shuffle=True, random_state=1), scoring='neg_mean_squared_error')
    mean_score = score.mean()
    mean_rmse = np.sqrt(-mean_score)

    return mean_rmse

study = optuna.create_study(direction="minimize")

study.optimize(lambda trial: objective(trial, X_df_train_fe, y_df_train), n_trials=50, show_progress_bar=True)


  0%|          | 0/50 [00:00<?, ?it/s]

Best trial: 42. Best value: 29233.3: 100%|██████████| 50/50 [1:25:40<00:00, 102.81s/it]


## Best scores

In [ ]:
# with cv tuning and log transformed target

print("Best RMSE:", study.best_value)
print("Best params:", study.best_params)


Best RMSE: 29233.32080975432
Best params: {'max_leaf_nodes': 130, 'max_depth': 3, 'learning_rate': 0.05297067443436627, 'max_iter': 1953, 'min_samples_leaf': 34, 'l2_regularization': 3.3580558110678207, 'max_features': 0.8393463201416943, 'max_bins': 230}


In [ ]:
# with cv tuning

print("Best RMSE:", study.best_value)
print("Best params:", study.best_params)


Best RMSE: 29069.475559231945
Best params: {'max_leaf_nodes': 133, 'max_depth': 6, 'learning_rate': 0.01668098332635971, 'max_iter': 611, 'min_samples_leaf': 109, 'l2_regularization': 4.00927060341554e-06, 'max_features': 0.6087452628047157, 'max_bins': 247}


In [50]:
# without cv
print("Best RMSE:", study.best_value)
print("Best params:", study.best_params)


Best RMSE: 25274.425148574053
Best params: {'max_leaf_nodes': 106, 'max_depth': 5, 'learning_rate': 0.027305231624249193, 'max_iter': 1196, 'min_samples_leaf': 108, 'l2_regularization': 4.482676560749479e-05, 'max_features': 0.6087340279698435, 'max_bins': 226}


In [ ]:
# with cv tuning and remove extreme y rows

print("Best RMSE:", study.best_value)
print("Best params:", study.best_params)


Best RMSE: 2525.415663651773
Best params: {'max_leaf_nodes': 59, 'max_depth': 10, 'learning_rate': 0.03649512270530141, 'max_iter': 1965, 'min_samples_leaf': 47, 'l2_regularization': 0.0002437490663992017, 'max_features': 0.7693286793633557, 'max_bins': 246}


## Prediction

In [153]:
df_test = raw_test.copy()


In [154]:
df_test.shape


(36000, 14)

In [155]:
# Fit on all dataset

X_df_train = df_train.drop(columns=[target_name])
y_df_train = df_train[target_name]

feature_engineering_pipeline.fit(X_df_train, y_df_train)


[SentenceEmbedding] Skipping fit (loading embeddings from ../data/02_intermediate/embeddings/desc_embeddings.db)
[SentenceEmbedding] Loading embeddings from ../data/02_intermediate/embeddings/desc_embeddings.db ...
[SentenceEmbedding] Loaded embeddings shape: (54000, 1025)


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('cat_feature_eng', ...), ('add_new_features', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('impute_MaritalStatus', ...), ('encode_cat_features', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid sea

In [157]:
X_df_train_fe = feature_engineering_pipeline.transform(X_df_train)


[SentenceEmbedding] Loading embeddings from ../data/02_intermediate/embeddings/desc_embeddings.db ...
[SentenceEmbedding] Loaded embeddings shape: (54000, 1025)


In [158]:
X_df_train_fe.shape


(54000, 124)

In [159]:
df_test_fe = feature_engineering_pipeline.transform(df_test)


[SentenceEmbedding] Loading embeddings from ../data/02_intermediate/embeddings/desc_embeddings.db ...
[SentenceEmbedding] Loaded embeddings shape: (54000, 1025)


In [160]:
best_model = HistGradientBoostingRegressor(
    **study.best_params,
    loss="squared_error",
    random_state=42
)

best_model.fit(X_df_train_fe, y_df_train)


,"loss loss: {'squared_error', 'absolute_error', 'gamma', 'poisson', 'quantile'}, default='squared_error'The loss function to use in the boosting process. Note that the""squared error"", ""gamma"" and ""poisson"" losses actually implement""half least squares loss"", ""half gamma deviance"" and ""half poissondeviance"" to simplify the computation of the gradient. Furthermore,""gamma"" and ""poisson"" losses internally use a log-link, ""gamma""requires ``y > 0`` and ""poisson"" requires ``y >= 0``.""quantile"" uses the pinball loss... versionchanged:: 0.23 Added option 'poisson'... versionchanged:: 1.1 Added option 'quantile'... versionchanged:: 1.3 Added option 'gamma'.",'squared_error'
,"quantile quantile: float, default=NoneIf loss is ""quantile"", this parameter specifies which quantile to be estimatedand must be between 0 and 1.",None
,"learning_rate learning_rate: float, default=0.1The learning rate, also known as *shrinkage*. This is used as amultiplicative factor for the leaves values. Use ``1`` for noshrinkage.",0.03649512270530141
,"max_iter max_iter: int, default=100The maximum number of iterations of the boosting process, i.e. themaximum number of trees.",1965
,"max_leaf_nodes max_leaf_nodes: int or None, default=31The maximum number of leaves for each tree. Must be strictly greaterthan 1. If None, there is no maximum limit.",59
,"max_depth max_depth: int or None, default=NoneThe maximum depth of each tree. The depth of a tree is the number ofedges to go from the root to the deepest leaf.Depth isn't constrained by default.",10
,"min_samples_leaf min_samples_leaf: int, default=20The minimum number of samples per leaf. For small datasets with lessthan a few hundred samples, it is recommended to lower this valuesince only very shallow trees would be built.",47
,"l2_regularization l2_regularization: float, default=0The L2 regularization parameter penalizing leaves with small hessians.Use ``0`` for no regularization (default).",0.0002437490663992017
,"max_features max_features: float, default=1.0Proportion of randomly chosen features in each and every node split.This is a form of regularization, smaller values make the trees weakerlearners and might prevent overfitting.If interaction constraints from `interaction_cst` are present, only allowedfeatures are taken into account for the subsampling... versionadded:: 1.4",0.7693286793633557
,"max_bins max_bins: int, default=255The maximum number of bins to use for non-missing values. Beforetraining, each feature of the input array `X` is binned intointeger-valued bins, which allows for a much faster training stage.Features with a small number of unique values may use less than``max_bins`` bins. In addition to the ``max_bins`` bins, one more binis always reserved for missing values. Must be no larger than 255.",246
,"categorical_features categorical_features: array-like of {bool, int, str} of shape (n_features) or shape (n_categorical_features,), default='from_dtype'Indicates the categorical features.- None : no feature will be considered categorical.- boolean array-like : boolean mask indicating categorical features.- integer array-like : integer indices indicating categorical features.- str array-like: names of categorical features (assuming the training data has feature names).- `""from_dtype""`: dataframe columns with dtype ""category"" are considered to be categorical features. The input must be an object exposing a ``__dataframe__`` method such as pandas or polars DataFrames to use this feature.For each categorical feature, there must be at most `max_bins` uniquecategories. Negative values for categorical features encoded as numericdtypes are treated as missing values. All categorical values areconverted to floating point numbers. This means that categorical valuesof 1.0 and 1 are treated as the same category.Read more in the :ref:`User Guide ` and:ref:`sphx_glr_auto_examples_ensemble_plot_gradient_boosting_categorical.py`... versionadded:: 0.24.. versionchanged:: 1.2 Adde

### Save best model

In [162]:
import joblib

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

joblib.dump(best_model, f"../data/06_models/model_{timestamp}.pkl")


['../data/06_models/model_20260525_021139.pkl']

In [ ]:
# removed extreme y value rows in train

train_preds = best_model.predict(X_df_train_fe)
train_rmse = root_mean_squared_error(y_df_train, train_preds)

print("Train RMSE:", train_rmse)


Train RMSE: 25850.950882652403


In [ ]:
# log transformed y
train_preds = best_model.predict(X_df_train_fe)
train_rmse = root_mean_squared_error(y_df_train, train_preds)

print("Train RMSE:", train_rmse)


Train RMSE: 28166.545782787776


In [ ]:
train_preds = best_model.predict(X_df_train_fe)
train_rmse = root_mean_squared_error(y_df_train, train_preds)

print("Train RMSE:", train_rmse)


Train RMSE: 27935.05491783373


In [163]:
test_preds = best_model.predict(df_test_fe)


In [164]:
test_preds


array([15578.3961461 ,  6242.65975421, 33012.06798542, ...,
       11705.01264098, 11035.36663675,  1877.39558745], shape=(36000,))

In [165]:
final_preds = df_test[["ClaimNumber"]]
final_preds["UltimateIncurredClaimCost"] = test_preds


In [166]:
final_preds


,ClaimNumber,UltimateIncurredClaimCost
0,WC8145235,15578.396146
1,WC2005111,6242.659754
2,WC6899143,33012.067985
3,WC5502023,2055.584605
4,WC4785156,8730.297416
...,...,...
35995,WC9666858,20294.947161
35996,WC4800526,4200.055266
35997,WC3360567,11705.012641
35998,WC7491778,11035.366637


In [167]:
final_preds.to_csv("../data/07_model_output/predictions_03.csv", index=False)
